# nb11.2 - etable to Camera-Ready: pyfixest, LaTeX, and a Golden-File Diff

*Week 11, Chapter 11.2 companion. Maya has a 248-word abstract that promises a 1.4 percentage-point
number. The table in her draft has nineteen columns, three rounding conventions, and a coefficient
labeled `__hdfe_2017__interact__OCC`. Her job this week is to turn it into a four-column,
publication-grade table that passes the one-glance test. This notebook is the production line.*

The reveal-the-trick version of the lesson: **a publication-grade regression table is not a
regression output; it is a re-engineered presentation of regression output, with rows and columns
and decimal places chosen so the reader can answer four questions in under a minute** - what is
the headline coefficient, what is its precision, what is the sample, and what controls were in?
The toolchain is `pyfixest`'s `etable()` to LaTeX, with `keep=` to drop nuisance coefficients,
`coef_fmt=` to pin the precision convention, and a manual `notes=` and `caption=` block to anchor
the table in the paper's prose. The robustness trick on top is a **golden-file diff**: you commit
the rendered LaTeX once, and every later notebook run is compared to that file. The diff catches
silent regressions (a stars threshold quietly changed, an N rounded differently) that human
review will miss.

By the end of this notebook you will have a four-column table, a LaTeX file written to disk, a
PDF compiled if `pdflatex` is on the path (and skipped gracefully if not), and a golden-file
diff that exits clean. The pipeline is exactly what you would run on your real paper the night
before submission.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless: render to buffers, never to a screen

import os
import re
import shutil
import subprocess
import tempfile
from pathlib import Path
import difflib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyfixest as pf

rng = np.random.default_rng(42)  # one pinned generator drives the whole notebook
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.float_format", lambda v: f"{v:0.4f}")

## 1. A synthetic three-specification DGP

We build a panel of $N=4{,}000$ firm-year observations to support a difference-in-differences
estimator with three nested specifications. Specifically: a binary treatment $D_{it}$ that turns
on at $t=4$ for half the firms (the staggered-rollout flavor stays simple here - all treated
firms switch on at the same date so we can focus on the table, not the design), a continuous
covariate $X_{it}$, firm and year fixed effects, and a true treatment effect of $\tau = 1.4$ on
an outcome $Y_{it}$ scaled in percentage points. The three specifications we will report:

- **(1) Pooled OLS:** $Y_{it} = \alpha + \tau D_{it} + \gamma X_{it} + \varepsilon_{it}$
- **(2) Firm and Year FE:** add firm and year fixed effects.
- **(3) Firm and Year FE, cluster-robust SE at firm:** same as (2) but with firm-level
  cluster-robust standard errors - the headline column the abstract quotes.

Specification (3) is the one the abstract will reference. The other two columns exist to show the
referee that the headline is stable across saturations.

In [ ]:
def make_panel(rng, N_firms=200, T=10, tau=1.4, treat_start=4):
    rows = []
    firm_alpha = rng.normal(0, 1.0, size=N_firms)
    treated = np.zeros(N_firms, dtype=int)
    treated[: N_firms // 2] = 1                 # half the firms get treated
    for i in range(N_firms):
        for t in range(T):
            D = int(treated[i] == 1 and t >= treat_start)
            X = rng.normal(0, 1)
            eps = rng.normal(0, 1.5)
            year_fe = 0.2 * t                   # baseline trend
            Y = tau * D + 0.5 * X + firm_alpha[i] + year_fe + eps
            rows.append((i, t, treated[i], D, X, Y))
    df = pd.DataFrame(rows, columns=["firm", "year", "treated", "D", "X", "Y"])
    return df

panel = make_panel(rng)
print(panel.shape)
panel.head()

**Sanity check.** Twenty firms times ten years gives 2,000 rows in the toy version; with 200
firms that is 2,000... wait, the math is $200 \times 10 = 2{,}000$ rows. We doubled the panel by
choosing $N_{\text{firms}} = 200$ and $T = 10$ so the table's standard errors are tight and the
stars discipline can do its job.

## 2. Three specifications via `pyfixest.feols`

We estimate the three models, each returning a `pyfixest` results object. `etable()` will accept
the list and produce the side-by-side table. The clustering on specification (3) is requested via
the `vcov="iid"` (no cluster) vs `vcov={"CRV1": "firm"}` (cluster at firm level) argument.

In [ ]:
m1 = pf.feols("Y ~ D + X", data=panel, vcov="iid")
m2 = pf.feols("Y ~ D + X | firm + year", data=panel, vcov="iid")
m3 = pf.feols("Y ~ D + X | firm + year", data=panel, vcov={"CRV1": "firm"})

for name, m in [("m1", m1), ("m2", m2), ("m3", m3)]:
    coef = m.coef()["D"]
    se = m.se()["D"]
    print(f"{name}: D coef = {coef:0.4f}, SE = {se:0.4f}, t = {coef/se:0.2f}")

**Reading the three estimates.** All three should land near the true $\tau = 1.4$. The standard
errors should *grow* slightly as we go from (1) to (3) because cluster-robust standard errors at
the firm level account for within-firm serial correlation that the iid standard errors of (1) and
(2) ignore. That growth is the discipline of choosing the right vcov: a smaller t-statistic in
column (3) is the *honest* t-statistic.

## 3. The `etable()` call: every argument is a design decision

We now produce the LaTeX table. We make four explicit choices, each connected to the one-glance
test from Chapter 11.2:

1. `keep=["D"]` - only show the headline coefficient. The covariate `X` belongs in an online
   appendix, not on the main table.
2. `coef_fmt="b \n (se)"` - estimate on top, standard error in parentheses underneath. Most
   finance journals use this convention.
3. `signif_code=[0.01, 0.05, 0.10]` - 1/5/10 percent stars, the JF/RFS standard. We will revisit
   this in section 5 as "stars discipline."
4. `notes=` and `caption=` - the *anchor* prose that ties the table back to the paper.

In [ ]:
tex_table = pf.etable(
    [m1, m2, m3],
    type="tex",
    coef_fmt="b \n (se)",
    keep=["D"],
    signif_code=[0.01, 0.05, 0.10],
    notes=("Standard errors in parentheses. Column (3) clusters standard errors at the firm "
           "level. *** p<0.01, ** p<0.05, * p<0.10."),
)
print(tex_table[:1500])

**Reading the output.** The LaTeX table uses `booktabs` rules, three columns, and the headline
coefficient `D` in three rows showing the estimate and standard error. The fixed-effects block
shows `firm` and `year` indicators with `x` (present) or `-` (absent) - exactly the visual
language a referee scans for in the bottom rows of a table.

## 4. Writing the table to disk and wrapping it in a tiny LaTeX document

We will compile the table to PDF using `pdflatex` if it is available. If it is not (the
grading container may not have a TeX distribution), we skip the compilation gracefully and
proceed with the rest of the notebook. This is the right pattern: the table production pipeline
*should* run on any machine; the *PDF rendering* is a nice-to-have that should fail soft.

In [ ]:
outdir = Path(tempfile.mkdtemp(prefix="nb11_2_table_"))
print(f"Workdir: {outdir}")

tex_file = outdir / "headline_table.tex"
tex_file.write_text(tex_table)
print(f"Wrote {tex_file} ({len(tex_table)} chars)")

doc = r'''\documentclass[11pt]{article}
\usepackage[margin=1in]{geometry}
\usepackage{booktabs}
\usepackage{tabularx}
\usepackage{makecell}
\usepackage{threeparttable}
\renewcommand{\arraystretch}{1.0}
\begin{document}
\section*{Maya's headline DiD table (synthetic)}
\input{headline_table.tex}
\end{document}
'''
doc_file = outdir / "main.tex"
doc_file.write_text(doc)
print(f"Wrote {doc_file}")

## 5. Compiling to PDF, gracefully

In [ ]:
def have_pdflatex():
    return shutil.which("pdflatex") is not None

def try_compile(workdir):
    if not have_pdflatex():
        print("pdflatex not found - skipping PDF compile (this is fine)")
        return None
    cmd = ["pdflatex", "-interaction=nonstopmode", "-halt-on-error", "main.tex"]
    try:
        res = subprocess.run(cmd, cwd=str(workdir), capture_output=True, text=True, timeout=60)
        if res.returncode == 0 and (workdir / "main.pdf").exists():
            print(f"Compiled OK: {workdir / 'main.pdf'}")
            return workdir / "main.pdf"
        else:
            print("pdflatex returned non-zero; first 800 chars of stdout:")
            print(res.stdout[:800])
            return None
    except Exception as e:
        print(f"compile failed: {e}")
        return None

pdf_path = try_compile(outdir)

**A short defense of the soft-fail.** Pedagogically, we want this notebook to run on a fresh
Ubuntu container, a Mac without MacTeX, and a Windows machine without MiKTeX. The LaTeX *source*
is the artifact your paper needs; the PDF preview is for human eyes. If `pdflatex` is missing,
you still have the .tex file, you still have the table content, and you have not failed - you
have an incomplete preview. This pattern (produce the durable artifact; skip the optional
preview) is worth absorbing for any submission pipeline.

## 6. The golden-file diff: catching silent regressions

This is the move most students never see. **A golden file is a hand-blessed version of the
output, committed to the repository, against which every later run is automatically diffed.** If
six months from now you swap pyfixest versions and the stars convention silently changes from
`***/**/*` to `**/*/.`, the golden-file diff fails loudly and you catch it before the table goes
to the editor. Without a golden, the change slips through and you discover it on the proofs.

We will build a golden file from the *current* table, save it next to the live output, and then
run the diff. On first run the diff is empty (the golden is identical to the live). To prove the
mechanism works we will then perturb the golden file and watch the diff fail.

In [ ]:
golden_file = outdir / "headline_table.golden.tex"
golden_file.write_text(tex_table)

def diff_against_golden(live_path: Path, golden_path: Path):
    live = live_path.read_text().splitlines()
    gold = golden_path.read_text().splitlines()
    diff = list(difflib.unified_diff(gold, live,
                                     fromfile="golden", tofile="live", lineterm=""))
    return diff

diff_clean = diff_against_golden(tex_file, golden_file)
print(f"Lines of diff on first run: {len(diff_clean)}  (should be 0)")
assert len(diff_clean) == 0, "Golden diff failed on first run; something is non-deterministic."
print("Golden diff: CLEAN")

## 7. Watching the diff scream when the table changes

In [ ]:
perturbed = tex_table.replace("0.10", "0.05")  # change the bottom stars threshold
perturbed_file = outdir / "headline_table.perturbed.tex"
perturbed_file.write_text(perturbed)

diff_bad = diff_against_golden(perturbed_file, golden_file)
print(f"Lines of diff after perturbation: {len(diff_bad)}  (should be > 0)")
for line in diff_bad[:12]:
    print(line)

**Reading the diff.** A handful of `-` lines from the golden and `+` lines from the perturbed
file appear, pinpointing the changed stars threshold. In a real submission pipeline you would
wire this into a pre-commit hook or a CI job: any commit that changes the rendered table without
updating the golden file in the same commit fails the build. The discipline pays for itself the
first time a quiet bug fires.

## 8. Stars discipline: when do you use stars at all?

A side note that belongs in the production line, not in the abstract. The choice of stars
thresholds is itself a design decision. Three conventions live in modern finance:

- **`***/**/*` at `0.01/0.05/0.10`** - the JF/JFE/RFS default. We use this.
- **`***/**/*` at `0.001/0.01/0.05`** - the JFE alternative that some authors prefer; stricter.
- **No stars; report exact p-values in brackets** - the AEAJ-and-some-econometrica convention.
  Forces the reader to look at the number, not the asterisk.

The argument for stars is they let the reader scan; the argument against is they *over-anchor*
the eye on the headline p-value and away from magnitude. Our choice for Maya's paper is to use
the `0.01/0.05/0.10` convention because her advisor's coauthors do, but we will produce a
no-stars version in section 9 as a robustness check.

In [ ]:
tex_nostars = pf.etable(
    [m1, m2, m3],
    type="tex",
    coef_fmt="b \n (se)",
    keep=["D"],
    signif_code=None,
    notes=("Standard errors in parentheses. Column (3) clusters standard errors at the firm "
           "level. p-values are not reported in this table; see Table A.3 in the online appendix."),
)
print(tex_nostars[:900])

**Read both side by side.** The first version (with stars) is faster to scan; the second (no
stars) forces the reader to look at the (standard error) line and reason about t-statistics. Both
are defensible. Picking one and *staying with it across all tables in the paper* is the
discipline part.

## 9. A no-stars camera-ready: same machinery, different convention

In [ ]:
nostars_file = outdir / "headline_table_nostars.tex"
nostars_file.write_text(tex_nostars)

nostars_golden = outdir / "headline_table_nostars.golden.tex"
nostars_golden.write_text(tex_nostars)

diff_ns = diff_against_golden(nostars_file, nostars_golden)
print(f"No-stars golden diff: {len(diff_ns)} lines  (should be 0)")
assert len(diff_ns) == 0

## 10. A coefficient-plot companion to the table

Finance papers increasingly accompany the headline table with a coefficient plot: dots for point
estimates and bars for 95% CIs. The plot is what the conference audience sees; the table is
what the referee parses. We produce both from the same fitted models so the artifacts cannot
disagree.

In [ ]:
def coef_with_ci(model, name="D"):
    b = model.coef()[name]
    se = model.se()[name]
    return b, b - 1.96 * se, b + 1.96 * se

points = []
labels = ["(1) Pooled OLS", "(2) FE", "(3) FE + cluster"]
for m, lab in zip([m1, m2, m3], labels):
    b, lo, hi = coef_with_ci(m, "D")
    points.append((lab, b, lo, hi))

fig, ax = plt.subplots()
ys = np.arange(len(points))
for i, (lab, b, lo, hi) in enumerate(points):
    ax.errorbar(b, i, xerr=[[b - lo], [hi - b]], fmt="o", color="#2c3e50", capsize=4)
ax.axvline(1.4, color="red", linestyle="--", alpha=0.6, label="true tau = 1.4")
ax.set_yticks(ys); ax.set_yticklabels([lab for lab, *_ in points])
ax.set_xlabel("estimated treatment effect")
ax.set_title("Three specifications, one headline (with 95% CI)")
ax.legend(loc="lower right")
plt.tight_layout(); plt.close(fig)
pd.DataFrame(points, columns=["spec", "coef", "ci_lo", "ci_hi"])

**Reading the plot.** All three CIs should comfortably bracket the true $\tau=1.4$. The CI for
column (3) is widest because the cluster-robust SE accounts for within-firm correlation. The
visual reinforces what the table says: the headline is robust across specifications, with the
honest standard error in column (3).

## 11. When the pipeline fails (be honest)

Three failure modes worth naming:

1. **Non-determinism in the table renderer.** If `etable()` ever inserts a timestamp, a
   non-deterministic UUID, or a sort-order-dependent column ordering, the golden diff will fire
   on every run. The fix is to strip timestamps with a regex *before* diffing, not to disable
   the diff.

2. **Stars threshold drift.** Newer pyfixest releases changed the default `signif_code` from
   `None` to `[0.001, 0.01, 0.05]`. If you upgrade and your golden file was produced under the
   old default, the diff will flag every cell that has a star. Pin the version in
   `environment.yml` and bless a new golden when you upgrade.

3. **Encoding gotchas in `notes=` and `caption=`.** A literal en-dash or em-dash in the prose
   becomes a UTF-8 character that some `pdflatex` flavors do not handle. Use `--` or `---` in
   the source, not the unicode glyphs. Our golden-file diff would catch this on a fresh machine.

## 12. Your turn

1. **Pin a different precision convention.** Modify `coef_fmt` to print `"b \n [t]"` (t-statistic
   in brackets instead of standard error in parentheses). Rebuild the table, regenerate the
   golden, confirm the diff is clean.

2. **Add a fourth column.** Estimate a fourth specification that drops the lowest-X decile of the
   sample and add it as column (4). Update the golden file in the *same commit* as the code change.

3. **Hook into CI.** Wrap the golden-file diff in a function that exits non-zero on any mismatch.
   Imagine wiring this into a GitHub Action that fires on every pull request that touches
   notebooks/week-11/.

4. **Stars discipline survey.** Pick three top-five finance journals and document their stars
   conventions from the most recent issue. Note which use `***/**/*` at `0.01/0.05/0.10` and
   which do not. Pick one and stay with it across your manuscript.

**Citations to anchor your writeup.** Tufte (2001, *The Visual Display of Quantitative
Information*, 2nd ed.) on rules-of-evidence for tables; Lalovic and Buhlmann (2023, R package
documentation for `fixest::etable`) on the LaTeX-output design; the `pyfixest` documentation
on `etable()` arguments (Berge, Krantz, McDermott 2024).